# ReAct RAG: A Bounded Retrieval Tool Loop

| Field | Value |
|---|---|
| Stage | LangGraph and agentic RAG |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A retrieval-tool ReAct loop must validate search actions, preserve citations, and stop within a fixed budget.

## 30-Second Summary

This notebook builds an offline LangGraph ReAct loop with one retrieval tool. The planner searches for paid-leave evidence, observes the cited result, and answers in one tool step; unsupported questions abstain without a call.

## Why This Matters

Turning retrieval into a tool gives an agent choice, but also introduces loop, argument, citation, and authorization failure modes that a fixed chain avoids.

## Scope

| Covers | Does not cover |
|---|---|
| Structured search action, tool validation, citation, abstention, step budget | Hosted planner, web search, multiple mutating tools, hidden reasoning |


## Mental Model

```text
question -> plan search(query) -> retrieval observation -> plan final answer -> END
```


In [1]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

DOCUMENTS = {"leave": "Employees receive twenty days of paid leave each year."}
MAX_STEPS = 2

class State(TypedDict, total=False):
    question: str
    steps: int
    action: dict
    observation: dict
    answer: str


## How It Works

The planner proposes a typed `search` action only for an in-scope question. The action node validates the tool and query, returns content plus source ID, increments the budget, and loops once for answer construction.


## Baseline

A direct retrieval chain always searches, even for unsupported questions. It lacks a decision boundary and tool-step terminal reason.


In [2]:
def always_search(question: str) -> dict:
    return {"content": DOCUMENTS["leave"], "source": "leave"}

always_search("Will it rain tomorrow?")


{'content': 'Employees receive twenty days of paid leave each year.',
 'source': 'leave'}

## Technique Implementation

The deterministic planner stands in for an LLM while preserving the operational contract. It emits actions, not free-form instructions, and never exposes hidden reasoning.


In [3]:
def plan(state: State) -> State:
    if state.get("observation"):
        item = state["observation"]
        return {"answer": f"Employees receive twenty days of paid leave [{item['source']}]."}
    if state.get("steps", 0) >= MAX_STEPS:
        return {"answer": "I stopped at the tool-step budget."}
    if "leave" in state["question"].lower() or "holiday" in state["question"].lower():
        return {"action": {"tool": "search", "query": "paid leave"}}
    return {"answer": "I cannot answer with the available retrieval tool."}

def act(state: State) -> State:
    action = state["action"]
    if action != {"tool": "search", "query": "paid leave"}:
        return {"answer": "Search action validation failed.", "action": {}}
    return {"observation": {"content": DOCUMENTS["leave"], "source": "leave"}, "steps": state.get("steps", 0) + 1, "action": {}}

def route(state: State) -> str:
    return "finish" if state.get("answer") else "act"


## Controlled Experiment

The answerable trace must be `plan → act → plan`, contain one search, and cite `leave`. The unsupported trace must terminate after the first plan.


In [4]:
builder = StateGraph(State)
builder.add_node("plan", plan)
builder.add_node("act", act)
builder.add_edge(START, "plan")
builder.add_conditional_edges("plan", route, {"finish": END, "act": "act"})
builder.add_edge("act", "plan")
agent = builder.compile()

answerable_trace = list(agent.stream({"question": "How much paid leave do employees get?", "steps": 0}, stream_mode="updates"))
answerable = agent.invoke({"question": "How much paid leave do employees get?", "steps": 0})
unsupported_trace = list(agent.stream({"question": "Will it rain tomorrow?", "steps": 0}, stream_mode="updates"))
{"answerable": answerable, "answerable_nodes": [next(iter(event)) for event in answerable_trace], "unsupported_nodes": [next(iter(event)) for event in unsupported_trace]}


{'answerable': {'question': 'How much paid leave do employees get?',
  'steps': 1,
  'action': {},
  'observation': {'content': 'Employees receive twenty days of paid leave each year.',
   'source': 'leave'},
  'answer': 'Employees receive twenty days of paid leave [leave].'},
 'answerable_nodes': ['plan', 'act', 'plan'],
 'unsupported_nodes': ['plan']}

## Evaluation

The leave question uses one validated search and answers with `[leave]`. The weather question terminates without retrieval. Both traces are bounded and inspectable.


In [5]:
assert answerable["steps"] == 1 and "[leave]" in answerable["answer"]
assert [next(iter(event)) for event in answerable_trace] == ["plan", "act", "plan"]
assert [next(iter(event)) for event in unsupported_trace] == ["plan"]
assert "cannot answer" in next(iter(unsupported_trace[0].values()))["answer"]
print("Retrieval-tool ReAct checks passed.")


Retrieval-tool ReAct checks passed.


## Decision Guide

| Need | Pattern |
|---|---|
| Retrieval always required | Fixed two-step RAG |
| Retrieval optional among few tools | Bounded ReAct |
| Known branching workflow | Explicit graph |
| Unsupported question | Abstain |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Repeated searches | No observation/step rule | Budget and duplicate-action guard |
| Citation lost | Tool returns text only | Typed result with source ID |
| Prompt injects tool call | Free-form action | Schema, allowlist, authorization |
| Weather retrieves leave policy | Missing abstention route | Scope classifier/fallback |


## Production Notes

### Observability
Trace action schema, query, ranked IDs, source citation, step count, latency, and terminal reason.

### Safety and Guardrails
Retrieval filters and tenant authorization apply inside the tool, not only in the prompt.

### Latency and Cost
Cap steps and duplicate calls; prefer fixed RAG when every valid question needs retrieval.


## Practice

Add a second document and require the agent to cite the exact retrieved source without exceeding two steps.

## Recall

Toggle - Recall: Why return source IDs from tools?
The final answer needs traceable evidence.

Toggle - Recall: When is ReAct unnecessary?
When the retrieval-and-answer sequence is fixed for every request.

## Sources

- [ReAct paper](https://arxiv.org/abs/2210.03629)
- [LangGraph workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the offline retrieval-tool contract | Add multi-tool conflicts and typed retrieval errors |
